# Improvement notebook v1 (run top to bottom)

This notebook is a **clean, linear improvement run** for the in-domain Memotion experiment.

## What it does
- keeps the same overall architecture
- uses a **separate output root** so older baseline/repro-fix outputs are not overwritten
- turns **sampler ON**
- uses a **smaller batch size**
- optimizes selection by **validation multimodal tuned-threshold Macro-F1**
- exports synchronized predictions, reports, table-ready CSV, and figure artifacts

## Output policy
This notebook writes to a new improvement folder under:

`D:\AI\outputs\runs_sarcasm_improve`

So the earlier frozen repro-fix baseline remains untouched.


In [1]:
import os
from huggingface_hub import login

HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✅ Hugging Face login completed via HF_TOKEN environment variable.")
else:
    print("ℹ️ HF_TOKEN environment variable not found.")
    print("   If gated model access fails, run:")
    print('   from huggingface_hub import login')
    print('   login(token="YOUR_HF_READ_TOKEN", add_to_git_credential=False)')


ℹ️ HF_TOKEN environment variable not found.
   If gated model access fails, run:
   from huggingface_hub import login
   login(token="YOUR_HF_READ_TOKEN", add_to_git_credential=False)


In [2]:
import os

os.environ["HF_HOME"] = r"D:\AI\hf_cache"
os.environ["TRANSFORMERS_CACHE"] = r"D:\AI\hf_cache\transformers"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [3]:
import os
HF_TOKEN = os.environ.get("HF_TOKEN")

In [4]:
import sys, os, torch, transformers
print("PYTHON:", sys.executable)
print("TORCH :", torch.__version__)
print("TFM   :", transformers.__version__)
print("CUDA  :", torch.cuda.is_available())
print("HF_HOME:", os.environ.get("HF_HOME"))
print("TRANSFORMERS_CACHE:", os.environ.get("TRANSFORMERS_CACHE"))

PYTHON: D:\AI\envs\sarcasm\python.exe
TORCH : 2.11.0+cu128
TFM   : 5.6.1
CUDA  : True
HF_HOME: D:\AI\hf_cache
TRANSFORMERS_CACHE: D:\AI\hf_cache\transformers


In [5]:
import transformers, huggingface_hub
print(transformers.__version__)
print(huggingface_hub.__version__)
from transformers import AutoTokenizer, AutoModel
print("✅ transformers import OK")

5.6.1
1.11.0
✅ transformers import OK


In [6]:
import os, re, glob, json, math, random, shutil, time
from pathlib import Path

os.environ.setdefault("HF_HOME", r"D:\AI\hf_cache")
os.environ.setdefault("TRANSFORMERS_CACHE", r"D:\AI\hf_cache\transformers")
HF_TOKEN = os.environ.get("HF_TOKEN")
print("HF_TOKEN set:", HF_TOKEN is not None)

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision import transforms
from PIL import Image

from transformers import AutoTokenizer, AutoModel, CLIPVisionModel

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ device:", device)

HF_TOKEN set: False
✅ device: cuda


In [7]:
DATA_ROOT = r"D:\AI\datasets\Memotion\memotion_dataset_7k"
OUTPUT_ROOT = r"D:\AI\outputs\runs_sarcasm_improve"

import os, glob

def _pick_first_existing(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None

label_candidates = []
for pat in ["labels*.csv", "labels*.xlsx", "labels*.xls", "**/labels*.csv", "**/labels*.xlsx", "**/labels*.xls"]:
    label_candidates += glob.glob(os.path.join(DATA_ROOT, pat), recursive=True)

label_candidates_sorted = sorted(label_candidates, key=lambda p: (0 if p.lower().endswith(".csv") else 1, len(p)))
CSV_PATH = label_candidates_sorted[0] if label_candidates_sorted else None

img_exts = (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif")
best_dir, best_count = None, 0
for root, dirs, files in os.walk(DATA_ROOT):
    c = sum(1 for f in files if f.lower().endswith(img_exts))
    if c > best_count:
        best_count = c
        best_dir = root

IMAGES_DIR = best_dir

os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("DATA_ROOT   :", DATA_ROOT)
print("CSV_PATH    :", CSV_PATH)
print("IMAGES_DIR  :", IMAGES_DIR, f"(images found: {best_count})")
print("OUTPUT_ROOT :", OUTPUT_ROOT)

if CSV_PATH is None:
    raise FileNotFoundError(
        "Could not find labels file under DATA_ROOT. Expected something like labels.csv / labels.xlsx. "
        f"DATA_ROOT={DATA_ROOT}"
    )
if IMAGES_DIR is None or best_count == 0:
    raise FileNotFoundError(
        "Could not find any images under DATA_ROOT. Please verify the dataset contains image files "
        "(.jpg/.png) and update DATA_ROOT accordingly."
    )

DATA_ROOT   : D:\AI\datasets\Memotion\memotion_dataset_7k
CSV_PATH    : D:\AI\datasets\Memotion\memotion_dataset_7k\labels.csv
IMAGES_DIR  : D:\AI\datasets\Memotion\memotion_dataset_7k\images (images found: 6991)
OUTPUT_ROOT : D:\AI\outputs\runs_sarcasm_improve


In [8]:

OUTPUT_ROOT = OUTPUT_ROOT
os.makedirs(OUTPUT_ROOT, exist_ok=True)

LR_LIST = [3e-5]
WD_LIST = [0.05]
DROPOUT_LIST = [0.5]
BATCH_LIST = [8]
EPOCHS_LIST = [8]
SELECT_BEST_BY = 'macro_f1_tuned_multimodal'

KEEP_ONLY_BEST = True
SAVE_FP16_WEIGHTS = True
KEEP_PREDICTIONS_FOR_BEST = True
COMPRESS_PREDICTIONS_GZ = False

print('✅ Improvement tuning space:', len(LR_LIST)*len(WD_LIST)*len(DROPOUT_LIST)*len(BATCH_LIST)*len(EPOCHS_LIST))
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('SELECT_BEST_BY =', SELECT_BEST_BY)


✅ Improvement tuning space: 1
OUTPUT_ROOT = D:\AI\outputs\runs_sarcasm_improve
SELECT_BEST_BY = macro_f1_tuned_multimodal


In [9]:
if 'CSV_PATH' in globals() and CSV_PATH:
    if not os.path.exists(CSV_PATH):
        raise FileNotFoundError(f"CSV_PATH not found: {CSV_PATH}")
    csv_path = CSV_PATH
else:
    csv_path = None

def find_first(patterns):
    for pat in patterns:
        hits = glob.glob(pat, recursive=True)
        if hits:
            hits_sorted = sorted(hits, key=lambda x: (("test" in x.lower()) + ("sample" in x.lower()), len(x)))
            return hits_sorted[0]
    return None

if csv_path is None:
    csv_path = find_first([
        "/kaggle/input/**/labels.csv",
        "/kaggle/input/**/train*.csv",
        "/kaggle/input/**/Train*.csv",
        "/kaggle/input/**/*memotion*/*.csv",
        "/kaggle/input/**/*.csv",
    ])
if csv_path is None:
    raise FileNotFoundError("Could not auto-find Memotion CSV under /kaggle/input. Attach dataset and retry.")

print("✅ Using CSV:", csv_path)
memotion_df = pd.read_csv(csv_path)
print("memotion_df shape:", memotion_df.shape)
print("Columns:", memotion_df.columns.tolist())


if 'IMAGES_DIR' in globals() and IMAGES_DIR:
    if not os.path.isdir(IMAGES_DIR):
        raise FileNotFoundError(f"IMAGES_DIR not found: {IMAGES_DIR}")
    images_dir = IMAGES_DIR
else:
    img_dirs = []
    for pat in ["/kaggle/input/**/images", "/kaggle/input/**/Images", "/kaggle/input/**/img", "/kaggle/input/**/Imgs", "/kaggle/input/**/memes"]:
        img_dirs.extend(glob.glob(pat, recursive=True))
    img_dirs = [d for d in img_dirs if os.path.isdir(d)]
    if not img_dirs:
        raise FileNotFoundError("Could not find an images directory under /kaggle/input. Check dataset mount.")
    images_dir = sorted(img_dirs, key=len)[0]
print("✅ Using images_dir:", images_dir)


if "image_name" not in memotion_df.columns:
    img_cands = [c for c in memotion_df.columns if re.search(r"(image|file|meme)", c, re.I)]
    if not img_cands:
        raise KeyError("No image filename column found.")
    memotion_df["image_name"] = memotion_df[img_cands[0]].astype(str)


if "binary_label" not in memotion_df.columns:
    use_col = "sarcasm" if "sarcasm" in memotion_df.columns else None
    if use_col is None:
        sarc_cands = [c for c in memotion_df.columns if re.search(r"sarc", c, re.I)]
        use_col = sarc_cands[0] if sarc_cands else None
    if use_col is None:
        raise KeyError("Could not infer sarcasm label column.")
    s = memotion_df[use_col].astype(str).str.strip().str.lower()
    is_num = s.str.fullmatch(r"-?\d+(\.\d+)?").fillna(False)
    binary = np.zeros(len(s), dtype=int)
    if is_num.any():
        num_vals = pd.to_numeric(s[is_num], errors="coerce").fillna(0).values
        binary[is_num.values] = (num_vals != 0).astype(int)
    neg_tokens = {"0","none","not_sarcastic","not sarcastic","no","na","nan","null","non-sarcastic","notsarcastic","not-sarcastic"}
    mask_str = (~is_num.values)
    if mask_str.any():
        binary[mask_str] = (~s[mask_str].isin(neg_tokens)).astype(int)
    memotion_df["binary_label"] = binary
    print(f"✅ Created binary_label from '{use_col}' with robust mapping.")
else:
    memotion_df["binary_label"] = pd.to_numeric(memotion_df["binary_label"], errors="coerce").fillna(0).astype(int)

print("binary_label distribution:\n", memotion_df["binary_label"].value_counts())

TEXT_COL = "text_corrected" if "text_corrected" in memotion_df.columns else ("text_ocr" if "text_ocr" in memotion_df.columns else None)
if TEXT_COL is None:
    raise KeyError("No usable text column found (text_corrected/text_ocr).")
print("✅ TEXT_COL:", TEXT_COL)

sample_paths = [os.path.join(images_dir, fn) for fn in memotion_df["image_name"].astype(str).head(50)]
exist_ct = sum([os.path.exists(p) for p in sample_paths])
print(f"✅ Sanity: {exist_ct}/50 sample images found.")


✅ Using CSV: D:\AI\datasets\Memotion\memotion_dataset_7k\labels.csv
memotion_df shape: (6992, 9)
Columns: ['Unnamed: 0', 'image_name', 'text_ocr', 'text_corrected', 'humour', 'sarcasm', 'offensive', 'motivational', 'overall_sentiment']
✅ Using images_dir: D:\AI\datasets\Memotion\memotion_dataset_7k\images
✅ Created binary_label from 'sarcasm' with robust mapping.
binary_label distribution:
 binary_label
1    5448
0    1544
Name: count, dtype: int64
✅ TEXT_COL: text_corrected
✅ Sanity: 50/50 sample images found.


In [10]:

train_df, temp_df = train_test_split(memotion_df, test_size=0.30, random_state=SEED, stratify=memotion_df["binary_label"])
val_df, test_df  = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["binary_label"])
print("Train/Val/Test:", train_df.shape, val_df.shape, test_df.shape)


Train/Val/Test: (4894, 10) (1049, 10) (1049, 10)


In [11]:

EMOJI_RE = re.compile("[" 
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FA6F"
    "\U0001FA70-\U0001FAFF"
    "\U00002700-\U000027BF"
    "\U00002600-\U000026FF"
"]+", flags=re.UNICODE)

def extract_emojis(text: str):
    if not isinstance(text, str): 
        return []
    return EMOJI_RE.findall(text)

emoji_counter = {}
for t in tqdm(train_df[TEXT_COL].astype(str).tolist(), desc="Building emoji vocab"):
    for e in extract_emojis(t):
        emoji_counter[e] = emoji_counter.get(e, 0) + 1

MAX_EMOJIS = 500
emoji_list = sorted(emoji_counter.items(), key=lambda x: x[1], reverse=True)[:MAX_EMOJIS]
emoji2id = {e:i+1 for i,(e,_) in enumerate(emoji_list)}  # 0=PAD
print("✅ emoji vocab size:", len(emoji2id))

def emojis_to_ids(text: str, max_k=16):
    ems = extract_emojis(text)
    ids = [emoji2id.get(e, 0) for e in ems][:max_k]
    if len(ids) < max_k:
        ids += [0]*(max_k-len(ids))
    return ids


Building emoji vocab:   0%|          | 0/4894 [00:00<?, ?it/s]

✅ emoji vocab size: 3


In [12]:


import os, re, json, math
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

from transformers import AutoTokenizer


TEXT_MODEL = "ai4bharat/indic-bert"
tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL,
    token=HF_TOKEN,
    use_fast=False,
)


resnet_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

clip_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466,0.4578275,0.40821073],
                         std=[0.26862954,0.26130258,0.27577711]),
])

class MemeDataset(Dataset):
    def __init__(self, df, images_dir, text_col, max_len=64, max_emojis=16):
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.text_col = text_col
        self.max_len = max_len
        self.max_emojis = max_emojis

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        img_path = os.path.join(self.images_dir, str(r["image_name"]))
        y = int(r["binary_label"])
        text = str(r[self.text_col])

        ok = 1
        try:
            img = Image.open(img_path).convert("RGB")
            pixel_resnet = resnet_transform(img)
            pixel_clip   = clip_transform(img)
        except Exception:
            pixel_resnet = torch.zeros(3,224,224)
            pixel_clip   = torch.zeros(3,224,224)
            ok = 0

        enc = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        eids = torch.tensor(emojis_to_ids(text, self.max_emojis), dtype=torch.long)

        return {
            "pixel_resnet": pixel_resnet,
            "pixel_clip": pixel_clip,
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "emoji_ids": eids,
            "label": torch.tensor(y, dtype=torch.float32),
            "ok": torch.tensor(ok, dtype=torch.long)
        }

def collate_fn(batch):
    batch = [b for b in batch if int(b["ok"]) == 1]
    if len(batch) == 0:
        return None
    out = {}
    for k in ["pixel_resnet","pixel_clip","input_ids","attention_mask","emoji_ids","label"]:
        out[k] = torch.stack([b[k] for b in batch], dim=0)
    return out

MAX_LEN = 64

train_ds = MemeDataset(train_df, images_dir, TEXT_COL, max_len=MAX_LEN)
val_ds   = MemeDataset(val_df, images_dir, TEXT_COL, max_len=MAX_LEN)
test_ds  = MemeDataset(test_df, images_dir, TEXT_COL, max_len=MAX_LEN)

print("✅ datasets ready")


y_tr = train_df["binary_label"].astype(int).values
class_counts = np.bincount(y_tr, minlength=2)
class_w = 1.0 / np.maximum(class_counts, 1)
sample_w = class_w[y_tr].astype(np.float32)

sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_w),
    num_samples=len(sample_w),
    replacement=True
)

print("Class counts:", class_counts, " | sampler ON")

✅ datasets ready
Class counts: [1081 3813]  | sampler ON


In [13]:


import torch, torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50, ResNet50_Weights
from transformers import AutoModel, CLIPVisionModel

class SarcasmPaperModel(nn.Module):
    def __init__(self, emoji_vocab_size, dropout=0.3):
        super().__init__()

        TEXT_MODEL = "ai4bharat/indic-bert"
        try:
            self.text_encoder = AutoModel.from_pretrained(TEXT_MODEL, token=HF_TOKEN, use_safetensors=True)
        except Exception:
            self.text_encoder = AutoModel.from_pretrained(TEXT_MODEL, token=HF_TOKEN)

        self.resnet = resnet50(weights=ResNet50_Weights.DEFAULT)
        self.resnet.fc = nn.Identity()        # 2048
        self.img_a_proj = nn.Linear(2048, 512)

        try:
            self.clip_vision = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32", use_safetensors=True)
        except Exception:
            self.clip_vision = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")

        self.img_b_proj = nn.Linear(self.clip_vision.config.hidden_size, 512)

        self.img_fuse = nn.Linear(1024, 768)

        self.emoji_emb = nn.Embedding(emoji_vocab_size + 1, 256, padding_idx=0)
        self.emoji_proj = nn.Linear(256, 768)

        self.attn_ti = nn.MultiheadAttention(embed_dim=768, num_heads=8, batch_first=True)
        self.attn_te = nn.MultiheadAttention(embed_dim=768, num_heads=8, batch_first=True)

        self.gate_ie = nn.Sequential(nn.Linear(768 + 768, 768), nn.Sigmoid())

        self.dropout = nn.Dropout(dropout)
        self.ln = nn.LayerNorm(768)

       
        self.head_text  = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 1))
        self.head_image = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 1))
        self.head_emoji = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 1))
        self.head_multi = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 1))

       
        self.sent_text  = nn.Linear(768, 1)
        self.sent_image = nn.Linear(768, 1)
        self.sent_emoji = nn.Linear(768, 1)

    def masked_emoji_mean(self, emoji_ids):
        emb = self.emoji_emb(emoji_ids)                   
        mask = (emoji_ids != 0).float().unsqueeze(-1)      
        denom = mask.sum(dim=1).clamp(min=1.0)
        pooled = (emb * mask).sum(dim=1) / denom
        return pooled

    def forward(self, pixel_resnet, pixel_clip, input_ids, attention_mask, emoji_ids):
        tout = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        Htext = tout.last_hidden_state
        htext = Htext[:, 0, :]

        fa2048 = self.resnet(pixel_resnet)
        fa = self.img_a_proj(fa2048)

        vout = self.clip_vision(pixel_values=pixel_clip)
        vb = vout.pooler_output if getattr(vout, "pooler_output", None) is not None else vout.last_hidden_state[:,0,:]
        fb = self.img_b_proj(vb)

        fimg = self.img_fuse(torch.cat([fa, fb], dim=-1))

        e = self.masked_emoji_mean(emoji_ids)
        femoji = self.emoji_proj(e)

        img_tok = fimg.unsqueeze(1)
        attn_ti, _ = self.attn_ti(Htext, img_tok, img_tok)
        H_ti = self.ln(Htext + self.dropout(attn_ti))

        emo_tok = femoji.unsqueeze(1)
        attn_te, _ = self.attn_te(H_ti, emo_tok, emo_tok)
        H_att = self.ln(H_ti + self.dropout(attn_te))
        h_att_cls = H_att[:, 0, :]

        g = self.gate_ie(torch.cat([fimg, femoji], dim=-1))
        fimg_g = g * fimg

        fused = self.ln(h_att_cls + fimg_g + femoji)

       
        logit_text  = self.head_text(htext).squeeze(-1)
        logit_image = self.head_image(fimg_g).squeeze(-1)
        logit_emoji = self.head_emoji(femoji).squeeze(-1)
        logit_multi = self.head_multi(fused).squeeze(-1)

       
        st_logit = self.sent_text(htext).squeeze(-1)
        si_logit = self.sent_image(fimg).squeeze(-1)
        se_logit = self.sent_emoji(femoji).squeeze(-1)

      
        st = torch.sigmoid(st_logit)
        si = torch.sigmoid(si_logit)
        se = torch.sigmoid(se_logit)
        incong = (torch.abs(st - si) + torch.abs(st - se) + torch.abs(si - se)) / 3.0

        return logit_text, logit_image, logit_emoji, logit_multi, incong, st_logit, si_logit, se_logit

model = SarcasmPaperModel(emoji_vocab_size=len(emoji2id), dropout=0.3).to(device)
print("✅ model ready (LOGITS + separate pixels)")

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
predictions.dense.bias           | UNEXPECTED |  | 
predictions.bias                 | UNEXPECTED |  | 
predictions.LayerNorm.bias       | UNEXPECTED |  | 
predictions.LayerNorm.weight     | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 
sop_classifier.classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_at

✅ model ready (LOGITS + separate pixels)


In [14]:

def gwo_optimize(branch_probs, y_true, n_wolves=20, iters=60, seed=SEED):
    rng = np.random.default_rng(seed)
    pop = rng.random((n_wolves, 4))
    pop = pop / pop.sum(axis=1, keepdims=True)

    def fitness(w):
        p = (branch_probs * w.reshape(1,4)).sum(axis=1)
        yhat = (p >= 0.5).astype(int)
        return f1_score(y_true, yhat, average="macro")

    scores = np.array([fitness(w) for w in pop])
    for t in range(iters):
        idx = np.argsort(scores)[::-1]
        alpha, beta, delta = pop[idx[0]], pop[idx[1]], pop[idx[2]]
        a = 2 - 2*(t/(iters-1+1e-9))
        for i in range(n_wolves):
            w = pop[i]
            r = rng.uniform(-1, 1, size=4)
            D_alpha = np.abs(alpha - w)
            D_beta  = np.abs(beta  - w)
            D_delta = np.abs(delta - w)
            w_new = (alpha - a*D_alpha + beta - a*D_beta + delta - a*D_delta)/3.0 + (a*0.1)*r
            w_new = np.clip(w_new, 0, 1)
            if w_new.sum() == 0:
                w_new = rng.random(4)
            w_new = w_new / w_new.sum()
            pop[i] = w_new
        scores = np.array([fitness(w) for w in pop])
    best_idx = np.argmax(scores)
    return pop[best_idx], float(scores[best_idx])


In [15]:

import os, json, numpy as np
from copy import deepcopy
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from pathlib import Path

PATIENCE = 3
CLIP_NORM = 1.0
NUM_WORKERS = 0

FREEZE_BACKBONES = True
PARTIAL_UNFREEZE = True

USE_SAMPLER = True
USE_POS_WEIGHT = False

W_AUX_SENT = 0.20
W_AUX_INCONG = 0.10

FOCAL_ALPHA = 0.60
FOCAL_GAMMA = 1.0

def make_loaders(batch_size):
    if USE_SAMPLER:
        tr = DataLoader(
            train_ds,
            batch_size=batch_size,
            sampler=sampler,
            shuffle=False,
            collate_fn=collate_fn,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            persistent_workers=False
        )
    else:
        tr = DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=True,
            collate_fn=collate_fn,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            persistent_workers=False
        )

    va = DataLoader(
        val_ds,
        batch_size=max(8, batch_size),
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=False
    )
    te = DataLoader(
        test_ds,
        batch_size=max(8, batch_size),
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=False
    )
    return tr, va, te

@torch.no_grad()
def eval_probs(model, loader):
    model.eval()
    ys, p_multi_all, br_all = [], [], []
    for batch in loader:
        if batch is None:
            continue
        batch = {k: v.to(device) for k, v in batch.items()}
        logit_text, logit_image, logit_emoji, logit_multi, incong, st_logit, si_logit, se_logit = model(
            batch["pixel_resnet"], batch["pixel_clip"],
            batch["input_ids"], batch["attention_mask"], batch["emoji_ids"]
        )
        y = batch["label"].detach().float().cpu().numpy()
        ys.append(y)

        p_text = torch.sigmoid(logit_text).detach().cpu().numpy()
        p_image = torch.sigmoid(logit_image).detach().cpu().numpy()
        p_emoji = torch.sigmoid(logit_emoji).detach().cpu().numpy()
        p_multi = torch.sigmoid(logit_multi).detach().cpu().numpy()

        p_multi_all.append(p_multi)
        br_all.append(np.stack([p_text, p_image, p_emoji, p_multi], axis=1))

    y_true = np.concatenate(ys) if ys else np.array([])
    p_multi = np.concatenate(p_multi_all) if p_multi_all else np.array([])
    br = np.concatenate(br_all) if br_all else np.zeros((0, 4))
    return y_true, p_multi, br

def metrics_from_probs(y_true, p, thr=0.5):
    if len(y_true) == 0:
        return {"acc": 0.0, "macro_f1": 0.0, "auc": 0.0}
    yhat = (p >= thr).astype(int)
    return {
        "acc": float(accuracy_score(y_true, yhat)),
        "macro_f1": float(f1_score(y_true, yhat, average="macro", zero_division=0)),
        "auc": float(roc_auc_score(y_true, p)) if len(np.unique(y_true)) == 2 else 0.0
    }

def tune_threshold_macro_f1(y_true, p, grid=None):
    if len(y_true) == 0:
        return 0.5, 0.0
    if grid is None:
        grid = np.linspace(0.05, 0.95, 37)
    best_t, best_f = 0.5, -1.0
    for t in grid:
        f = f1_score(y_true, (p >= t).astype(int), average="macro", zero_division=0)
        if f > best_f:
            best_f, best_t = float(f), float(t)
    return best_t, best_f

def save_best_state(model, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(out_dir, "best_model.pt"))

def focal_bce_with_logits(logits, targets, alpha=0.75, gamma=2.0):
    bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p = torch.sigmoid(logits)
    pt = targets * p + (1 - targets) * (1 - p)
    w = alpha * targets + (1 - alpha) * (1 - targets)
    loss = w * (1 - pt).pow(gamma) * bce
    return loss.mean()

def gwo_optimize(br_probs, y_true, n_wolves=18, iters=60, seed=42, metric="auc"):
    rng = np.random.default_rng(seed)
    if len(y_true) == 0:
        return np.array([0.25, 0.25, 0.25, 0.25], float)

    def score(w):
        w = np.clip(w, 1e-8, 1.0)
        w = w / w.sum()
        p = (br_probs * w.reshape(1, 4)).sum(axis=1)
        if metric == "auc":
            return roc_auc_score(y_true, p) if len(np.unique(y_true)) == 2 else 0.0
        t, _ = tune_threshold_macro_f1(y_true, p)
        return f1_score(y_true, (p >= t).astype(int), average="macro", zero_division=0)

    wolves = rng.random((n_wolves, 4))
    wolves = wolves / wolves.sum(axis=1, keepdims=True)
    alpha, beta, delta = wolves[0].copy(), wolves[1].copy(), wolves[2].copy()
    alpha_s = beta_s = delta_s = -1e9

    for t in range(iters):
        scores = np.array([score(w) for w in wolves])
        idx = np.argsort(scores)[::-1]
        wolves, scores = wolves[idx], scores[idx]
        if scores[0] > alpha_s:
            alpha, alpha_s = wolves[0].copy(), scores[0]
        if scores[1] > beta_s:
            beta, beta_s = wolves[1].copy(), scores[1]
        if scores[2] > delta_s:
            delta, delta_s = wolves[2].copy(), scores[2]

        a = 2 - 2 * (t / (iters - 1 + 1e-9))
        new = []
        for i in range(n_wolves):
            X = wolves[i].copy()

            def upd(Xp):
                r1, r2 = rng.random(4), rng.random(4)
                A = 2 * a * r1 - a
                C = 2 * r2
                D = np.abs(C * Xp - X)
                return Xp - A * D

            Xn = (upd(alpha) + upd(beta) + upd(delta)) / 3.0
            Xn = np.clip(Xn, 1e-8, 1.0)
            Xn = Xn / Xn.sum()
            new.append(Xn)
        wolves = np.vstack(new)

    w_best = np.clip(alpha, 1e-8, 1.0)
    w_best = w_best / w_best.sum()
    return w_best

CONFIG_DIR = Path(r"D:\AI\projects\sarcasm_project\configs")
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

IMPROVE_CFG_PATH = CONFIG_DIR / "IMPROVE_V4_CONFIG.json"
improve_cfg = {
    "lr": LR_LIST[0],
    "weight_decay": WD_LIST[0],
    "dropout": DROPOUT_LIST[0],
    "batch_size": BATCH_LIST[0],
    "epochs": EPOCHS_LIST[0],
    "focal_alpha": FOCAL_ALPHA,
    "focal_gamma": FOCAL_GAMMA,
    "sampler": USE_SAMPLER,
    "select_best_by": SELECT_BEST_BY
}
with open(IMPROVE_CFG_PATH, "w", encoding="utf-8") as f:
    json.dump(improve_cfg, f, indent=2)

LR = float(improve_cfg["lr"])
WD = float(improve_cfg["weight_decay"])
DROPOUT = float(improve_cfg["dropout"])
BS = int(improve_cfg["batch_size"])
EPOCHS = int(improve_cfg["epochs"])

run_name = (
    f"improve_v4_lr{LR:g}_wd{WD:g}_do{DROPOUT:g}_"
    f"bs{BS}_ep{EPOCHS}_fa{FOCAL_ALPHA:g}_fg{FOCAL_GAMMA:g}_"
    f"{'SAMPLER' if USE_SAMPLER else 'NOSAMPLER'}"
)
run_dir = os.path.join(OUTPUT_ROOT, run_name)
os.makedirs(run_dir, exist_ok=True)

with open(os.path.join(run_dir, "IMPROVE_V4_CONFIG.json"), "w", encoding="utf-8") as f:
    json.dump(improve_cfg, f, indent=2)

train_loader, val_loader, test_loader = make_loaders(BS)

model = SarcasmPaperModel(emoji_vocab_size=len(emoji2id), dropout=DROPOUT).to(device)

if FREEZE_BACKBONES:
    for p in model.text_encoder.parameters():
        p.requires_grad = False
    for p in model.resnet.parameters():
        p.requires_grad = False
    for p in model.clip_vision.parameters():
        p.requires_grad = False
    print("✅ Backbones frozen.")

if PARTIAL_UNFREEZE:
    if hasattr(model.text_encoder, "encoder") and hasattr(model.text_encoder.encoder, "layer"):
        for layer in model.text_encoder.encoder.layer[-2:]:
            for p in layer.parameters():
                p.requires_grad = True
    if hasattr(model.resnet, "layer4"):
        for p in model.resnet.layer4.parameters():
            p.requires_grad = True
    if hasattr(model.clip_vision, "vision_model") and hasattr(model.clip_vision.vision_model, "encoder"):
        blocks = model.clip_vision.vision_model.encoder.layers
        for blk in blocks[-2:]:
            for p in blk.parameters():
                p.requires_grad = True
    print("✅ Partial unfreeze enabled.")

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WD)

use_cuda_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler('cuda', enabled=use_cuda_amp)

best_val = -1e9
best_epoch = -1
no_improve = 0
best_state = None

history = []

for ep in range(1, EPOCHS + 1):
    model.train()
    losses = []
    pbar = tqdm(train_loader, desc=f"Improve Epoch {ep}/{EPOCHS}", leave=False)

    for batch in pbar:
        if batch is None:
            continue
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=use_cuda_amp):
            logit_text, logit_image, logit_emoji, logit_multi, incong, st_logit, si_logit, se_logit = model(
                batch["pixel_resnet"], batch["pixel_clip"],
                batch["input_ids"], batch["attention_mask"], batch["emoji_ids"]
            )

            y = batch["label"].float()
            loss_main = focal_bce_with_logits(logit_multi.float(), y, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)
            loss_aux_sent = (
                focal_bce_with_logits(st_logit.float(), y, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA) +
                focal_bce_with_logits(si_logit.float(), y, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA) +
                focal_bce_with_logits(se_logit.float(), y, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)
            ) / 3.0
            # BCE on probabilities is unsafe under autocast; compute this auxiliary term in FP32 outside autocast.
            with torch.amp.autocast('cuda', enabled=False):
                loss_aux_incong = torch.nn.functional.binary_cross_entropy(
                    incong.float().clamp(0, 1),
                    y.float()
                )
            loss = loss_main + W_AUX_SENT * loss_aux_sent + W_AUX_INCONG * loss_aux_incong

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()

        losses.append(loss.item())
        if len(losses) % 20 == 0:
            pbar.set_postfix(loss=float(np.mean(losses[-20:])))

    yv, pv, brv = eval_probs(model, val_loader)

    m_multi_05 = metrics_from_probs(yv, pv, thr=0.5)
    thr_multi, _ = tune_threshold_macro_f1(yv, pv)
    m_multi_best = metrics_from_probs(yv, pv, thr=thr_multi)

    w_best = gwo_optimize(brv, yv.astype(int), metric="auc")
    p_ens = (brv * w_best.reshape(1, 4)).sum(axis=1)
    m_ens_05 = metrics_from_probs(yv, p_ens, thr=0.5)
    thr_ens, _ = tune_threshold_macro_f1(yv, p_ens)
    m_ens_best = metrics_from_probs(yv, p_ens, thr=thr_ens)

    val_metric = m_multi_best["macro_f1"]

    epoch_row = {
        "epoch": ep,
        "train_loss": float(np.mean(losses)) if losses else None,
        "multi_05_acc": m_multi_05["acc"],
        "multi_05_macro_f1": m_multi_05["macro_f1"],
        "multi_05_auc": m_multi_05["auc"],
        "multi_best_thr": float(thr_multi),
        "multi_best_acc": m_multi_best["acc"],
        "multi_best_macro_f1": m_multi_best["macro_f1"],
        "multi_best_auc": m_multi_best["auc"],
        "ens_05_acc": m_ens_05["acc"],
        "ens_05_macro_f1": m_ens_05["macro_f1"],
        "ens_05_auc": m_ens_05["auc"],
        "ens_best_thr": float(thr_ens),
        "ens_best_acc": m_ens_best["acc"],
        "ens_best_macro_f1": m_ens_best["macro_f1"],
        "ens_best_auc": m_ens_best["auc"],
        "selection_metric": float(val_metric)
    }
    history.append(epoch_row)
    pd.DataFrame(history).to_csv(os.path.join(run_dir, "training_history.csv"), index=False)

    print(
        f"[VAL] ep={ep} loss={np.mean(losses):.4f} | "
        f"multi_best_macroF1={m_multi_best['macro_f1']:.4f} thr={thr_multi:.3f} | "
        f"multi_auc={m_multi_best['auc']:.4f} | "
        f"ens_best_macroF1={m_ens_best['macro_f1']:.4f} thr={thr_ens:.3f} | "
        f"ens_auc={m_ens_best['auc']:.4f}"
    )

    if val_metric > best_val + 1e-6:
        best_val = float(val_metric)
        best_epoch = ep
        no_improve = 0
        best_state = deepcopy(model.state_dict())

        snap = {
            "run": run_name,
            "best_epoch": int(best_epoch),
            "selection_metric": "val_multi_best_macro_f1",
            "val_multi_0_5": m_multi_05,
            "val_multi_thr": float(thr_multi),
            "val_multi_best": m_multi_best,
            "val_ens_0_5": m_ens_05,
            "val_ens_thr": float(thr_ens),
            "val_ens_best": m_ens_best,
            "gwo_weights": w_best.tolist(),
            "focal_alpha": float(FOCAL_ALPHA),
            "focal_gamma": float(FOCAL_GAMMA),
            "sampler": bool(USE_SAMPLER)
        }
        with open(os.path.join(run_dir, "best_metrics.json"), "w", encoding="utf-8") as f:
            json.dump(snap, f, indent=2)

        model.load_state_dict(best_state)
        save_best_state(model, run_dir)
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print("Early stopping.")
            break

print("✅ Improvement training done.")
print("Best epoch:", best_epoch)
print("Best validation selection metric:", best_val)
print("Run dir:", run_dir)


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
predictions.dense.bias           | UNEXPECTED |  | 
predictions.bias                 | UNEXPECTED |  | 
predictions.LayerNorm.bias       | UNEXPECTED |  | 
predictions.LayerNorm.weight     | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 
sop_classifier.classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_at

✅ Backbones frozen.
✅ Partial unfreeze enabled.


...

[VAL] ep=2 loss=0.2713 | multi_best_macroF1=0.4585 thr=0.300 | multi_auc=0.4305 | ens_best_macroF1=0.4378 thr=0.050 | ens_auc=0.5242


Improve Epoch 3/8:   0%|▏                                                              | 2/612 [00:00<01:15,  8.11it/s]D:\AI\envs\sarcasm\lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
                                                                                                                       

[VAL] ep=3 loss=0.2177 | multi_best_macroF1=0.4825 thr=0.175 | multi_auc=0.4444 | ens_best_macroF1=0.4378 thr=0.050 | ens_auc=0.5213


Improve Epoch 4/8:  23%|███████████▏                                     | 139/612 [00:14<00:46, 10.15it/s, loss=0.184]D:\AI\envs\sarcasm\lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
                                                                                                                       

[VAL] ep=4 loss=0.1857 | multi_best_macroF1=0.4866 thr=0.300 | multi_auc=0.4493 | ens_best_macroF1=0.4378 thr=0.050 | ens_auc=0.5240
Early stopping.
✅ Improvement training done.
Best epoch: 1
Best validation selection metric: 0.5064605247133178
Run dir: D:\AI\outputs\runs_sarcasm_improve\improve_v4_lr3e-05_wd0.05_do0.5_bs8_ep8_fa0.6_fg1_SAMPLER


## Improvement export

The next two cells:
- load the best checkpoint from this **improvement run**,
- recompute test probabilities in a synchronized way,
- write JSON / CSV / text outputs into the **same separate improvement folder**,
- generate publication-style confusion matrix, ROC, and PR figures with color bars.


In [16]:

from pathlib import Path
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve, auc,
    average_precision_score
)
from torch.utils.data import DataLoader

required_globals = ["eval_probs", "metrics_from_probs", "SarcasmPaperModel", "test_ds", "collate_fn", "emoji2id", "device", "run_dir"]
missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required notebook variables: {missing_globals}. Run all previous cells first.")

improve_run_dir = Path(run_dir)
if not improve_run_dir.exists():
    raise FileNotFoundError(f"Improvement run folder not found: {improve_run_dir}")

best_metrics_path = improve_run_dir / "best_metrics.json"
best_model_path = improve_run_dir / "best_model.pt"
config_path = improve_run_dir / "IMPROVE_V4_CONFIG.json"

for p in [best_metrics_path, best_model_path, config_path]:
    if not p.exists():
        raise FileNotFoundError(f"Required improvement artifact missing: {p}")

with open(best_metrics_path, "r", encoding="utf-8") as f:
    snap = json.load(f)
with open(config_path, "r", encoding="utf-8") as f:
    improve_cfg_local = json.load(f)

BS_LOCAL = int(improve_cfg_local.get("batch_size", 8))
test_loader_local = DataLoader(
    test_ds,
    batch_size=max(8, BS_LOCAL),
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True,
    persistent_workers=False
)

DROPOUT_LOCAL = float(improve_cfg_local.get("dropout", 0.5))
model_local = SarcasmPaperModel(emoji_vocab_size=len(emoji2id), dropout=DROPOUT_LOCAL).to(device)
state = torch.load(best_model_path, map_location=device)
model_local.load_state_dict(state)
model_local.eval()

yt, pt, brt = eval_probs(model_local, test_loader_local)
yt = np.asarray(yt).astype(int)
pt = np.asarray(pt).astype(float)
brt = np.asarray(brt).astype(float)

if brt.ndim != 2 or brt.shape[1] != 4:
    raise RuntimeError(f"Expected branch_probs with shape [N,4], got {brt.shape}")

p_text = brt[:, 0]
p_image = brt[:, 1]
p_emoji = brt[:, 2]
p_multi = brt[:, 3]

w_best = np.asarray(snap["gwo_weights"], dtype=float)
p_ens = (brt * w_best.reshape(1, 4)).sum(axis=1)

thr_multi_best = float(snap.get("val_multi_thr", 0.5))
thr_ens_best = float(snap.get("val_ens_thr", 0.5))

def metric_bundle(y_true, y_prob, thr):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_hat = (y_prob >= float(thr)).astype(int)
    out = {
        "acc": float(accuracy_score(y_true, y_hat)),
        "macro_f1": float(f1_score(y_true, y_hat, average="macro", zero_division=0)),
        "auc": float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) == 2 else None,
        "threshold": float(thr),
        "positive_rate": float(np.mean(y_hat))
    }
    return out

m_multi_05 = metric_bundle(yt, p_multi, 0.5)
m_multi_best = metric_bundle(yt, p_multi, thr_multi_best)
m_ens_05 = metric_bundle(yt, p_ens, 0.5)
m_ens_best = metric_bundle(yt, p_ens, thr_ens_best)

pred_df = pd.DataFrame({
    "y_true": yt,
    "p_text": p_text,
    "p_image": p_image,
    "p_emoji": p_emoji,
    "p_multi": p_multi,
    "p_ens": p_ens,
    "yhat_multi_thr_0_5": (p_multi >= 0.5).astype(int),
    "yhat_multi_thr_best": (p_multi >= thr_multi_best).astype(int),
    "yhat_ens_thr_0_5": (p_ens >= 0.5).astype(int),
    "yhat_ens_thr_best": (p_ens >= thr_ens_best).astype(int),
})
pred_df.to_csv(improve_run_dir / "predictions.csv", index=False)
pred_df.to_csv(improve_run_dir / "predictions_improve_v4.csv", index=False)

report_multi_best = classification_report(yt, (p_multi >= thr_multi_best).astype(int), digits=4, zero_division=0)
report_ens_best = classification_report(yt, (p_ens >= thr_ens_best).astype(int), digits=4, zero_division=0)
report_multi_05 = classification_report(yt, (p_multi >= 0.5).astype(int), digits=4, zero_division=0)
report_ens_05 = classification_report(yt, (p_ens >= 0.5).astype(int), digits=4, zero_division=0)

(improve_run_dir / "classification_report_multi_best.txt").write_text(report_multi_best, encoding="utf-8")
(improve_run_dir / "classification_report_ens_best.txt").write_text(report_ens_best, encoding="utf-8")
(improve_run_dir / "classification_report_multi_0_5.txt").write_text(report_multi_05, encoding="utf-8")
(improve_run_dir / "classification_report_ens_0_5.txt").write_text(report_ens_05, encoding="utf-8")

(improve_run_dir / "classification_report.txt").write_text(report_multi_best, encoding="utf-8")

final_eval = {
    "run_dir": str(improve_run_dir),
    "best_epoch": int(snap.get("best_epoch", -1)),
    "selection_metric": snap.get("selection_metric", "val_multi_best_macro_f1"),
    "thresholds": {
        "val_multi_thr": thr_multi_best,
        "val_ens_thr": thr_ens_best
    },
    "test_size": int(len(yt)),
    "test_pos_frac": float(np.mean(yt)) if len(yt) else None,
    "multi_thr_0_5": m_multi_05,
    "multi_thr_best": m_multi_best,
    "ens_thr_0_5": m_ens_05,
    "ens_thr_best": m_ens_best,
    "ensemble_weights": w_best.tolist()
}
with open(improve_run_dir / "final_eval_fixed.json", "w", encoding="utf-8") as f:
    json.dump(final_eval, f, indent=2)

table2 = pd.DataFrame([
    {"experiment": "improve_v4_multimodal_thr_0_5", **m_multi_05},
    {"experiment": "improve_v4_multimodal_thr_best", **m_multi_best},
    {"experiment": "improve_v4_ensemble_thr_0_5", **m_ens_05},
    {"experiment": "improve_v4_ensemble_thr_best", **m_ens_best},
])
table2.to_csv(improve_run_dir / "table2_in_domain_improve_v4.csv", index=False)

print("✅ Saved synchronized improvement predictions, reports, final_eval_fixed.json, and table2_in_domain_improve_v4.csv")
print(table2)


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
predictions.dense.bias           | UNEXPECTED |  | 
predictions.bias                 | UNEXPECTED |  | 
predictions.LayerNorm.bias       | UNEXPECTED |  | 
predictions.LayerNorm.weight     | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 
sop_classifier.classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_at

✅ Saved synchronized improvement predictions, reports, final_eval_fixed.json, and table2_in_domain_improve_v4.csv
                       experiment       acc  macro_f1      auc  threshold  \
0   improve_v4_multimodal_thr_0_5  0.677788  0.487686  0.50330      0.500   
1  improve_v4_multimodal_thr_best  0.647283  0.505995  0.50330      0.525   
2     improve_v4_ensemble_thr_0_5  0.220210  0.180469  0.48211      0.500   
3    improve_v4_ensemble_thr_best  0.779790  0.438136  0.48211      0.050   

   positive_rate  
0       0.829361  
1       0.755005  
2       0.000000  
3       1.000000  


In [17]:

from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, precision_recall_curve, auc, average_precision_score

required_plot_globals = ["improve_run_dir", "yt", "p_multi", "p_ens", "thr_multi_best", "thr_ens_best", "final_eval"]
missing_plot_globals = [x for x in required_plot_globals if x not in globals()]
if missing_plot_globals:
    raise RuntimeError(f"Missing variables for plotting: {missing_plot_globals}. Run the previous export cell first.")

improve_run_dir = Path(improve_run_dir)

def save_confmat(y_true, y_pred, title, out_path):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5.5, 4.8), dpi=220)
    im = ax.imshow(cm, interpolation="nearest")
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Count")
    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["0", "1"]); ax.set_yticklabels(["0", "1"])
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)

def save_roc(y_true, y_prob, title, out_path):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(5.5, 4.8), dpi=220)
    ax.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
    ax.plot([0, 1], [0, 1], linestyle="--")
    ax.set_title(title)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)

def save_pr(y_true, y_prob, title, out_path):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(5.5, 4.8), dpi=220)
    ax.plot(recall, precision, label=f"AP = {ap:.4f}")
    ax.set_title(title)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.legend(loc="lower left")
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)

save_confmat(yt, (p_multi >= thr_multi_best).astype(int),
             "Confusion Matrix (Improve v4 Multimodal, tuned threshold)",
             improve_run_dir / "confusion_matrix_multi_best.png")
save_roc(yt, p_multi, "ROC Curve (Improve v4 Multimodal)", improve_run_dir / "roc_multi.png")
save_pr(yt, p_multi, "PR Curve (Improve v4 Multimodal)", improve_run_dir / "pr_multi.png")

save_confmat(yt, (p_ens >= thr_ens_best).astype(int),
             "Confusion Matrix (Improve v4 Ensemble, tuned threshold)",
             improve_run_dir / "confusion_matrix_ens_best.png")
save_roc(yt, p_ens, "ROC Curve (Improve v4 Ensemble)", improve_run_dir / "roc_ens.png")
save_pr(yt, p_ens, "PR Curve (Improve v4 Ensemble)", improve_run_dir / "pr_ens.png")

for src_name, dst_name in [
    ("confusion_matrix_ens_best.png", "fig5_confusion_matrix_ensemble.png"),
    ("roc_ens.png", "fig5_roc_ensemble.png"),
    ("pr_ens.png", "fig5_pr_ensemble.png"),
]:
    src = improve_run_dir / src_name
    dst = improve_run_dir / dst_name
    if src.exists():
        import shutil
        shutil.copy2(src, dst)

manifest = {
    "run_dir": str(improve_run_dir),
    "generated_files": sorted([p.name for p in improve_run_dir.iterdir() if p.is_file()])
}
with open(improve_run_dir / "artifact_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("✅ Saved improvement confusion matrices, ROC, PR, and artifact_manifest.json")
print("Files:")
for name in manifest["generated_files"]:
    print(" -", name)


✅ Saved improvement confusion matrices, ROC, PR, and artifact_manifest.json
Files:
 - IMPROVE_V4_CONFIG.json
 - best_metrics.json
 - best_model.pt
 - classification_report.txt
 - classification_report_ens_0_5.txt
 - classification_report_ens_best.txt
 - classification_report_multi_0_5.txt
 - classification_report_multi_best.txt
 - confusion_matrix_ens_best.png
 - confusion_matrix_multi_best.png
 - fig5_confusion_matrix_ensemble.png
 - fig5_pr_ensemble.png
 - fig5_roc_ensemble.png
 - final_eval_fixed.json
 - pr_ens.png
 - pr_multi.png
 - predictions.csv
 - predictions_improve_v4.csv
 - roc_ens.png
 - roc_multi.png
 - table2_in_domain_improve_v4.csv
 - training_history.csv
